# Build an HTML report from ACS Census data

This notebook walks through scraping the ACS "Data" landing page
(`census.gov/programs-surveys/acs/data.html`) for context and links, pulling
real numbers from the Census Data API, and rendering both into a single
static HTML report — running each step live so you can see the actual data
at every stage.

It calls into the same modules used by the plain-script version of this
tutorial (`scrape_acs_resources.py`, `fetch_acs_data.py`, `generate_report.py`,
all in this folder) rather than duplicating their logic — this notebook is
the interactive walkthrough, they're the source of truth.

## 0. Setup

From `census_dashboard/`, if you haven't already:

```powershell
python -m venv venv
venv\Scripts\pip install -r requirements.txt
venv\Scripts\pip install ipykernel
venv\Scripts\python -m ipykernel install --user --name census-dashboard-venv --display-name "Python (census_dashboard venv)"
```

Then, in VS Code / Jupyter, select the **"Python (census_dashboard venv)"**
kernel for this notebook (top-right kernel picker).

Get a free Census API key (instant signup) and put it in a `.env` file next
to this notebook: <https://api.census.gov/data/key_signup.html>

```
CENSUS_API_KEY=your_key_here
```

Everything below still runs without a key — the report falls back to real,
bundled 2023 sample values and tells you plainly when it's doing that.

In [1]:
import pathlib

NOTEBOOK_DIR = pathlib.Path.cwd()
assert (NOTEBOOK_DIR / "scrape_acs_resources.py").exists(), (
    "Run this notebook from inside census_dashboard/ so it can import the sibling .py modules"
)

import scrape_acs_resources
import fetch_acs_data
import generate_report
from IPython.display import HTML, IFrame, display

print("Modules loaded from", NOTEBOOK_DIR)

Modules loaded from C:\Users\drdre\Healthy_Vets_Stuff\census_dashboard


## 1. Why this isn't a single scrape

Open `census.gov/programs-surveys/acs/data.html` in devtools before writing
any scraper and you'll find it's a **navigation hub, not a data page** — no
ACS numbers live in its HTML. Let's confirm that live: fetch the page and
look at its actual headings.

In [2]:
soup = scrape_acs_resources.fetch_page()

print("Page title:", soup.title.get_text(strip=True))
print("\nSection headings on the page:")
for h in soup.select("h1.cmp-title__text, h3.cmp-title__text"):
    print(" -", h.get_text(strip=True))

Page title: American Community Survey Data

Section headings on the page:
 - American Community Survey Data
 - American Community Survey Data
 - Get Started Accessing ACS Data
 - View Popular ACS Tables
 - Discover Popular ACS Data Resources


Three sections, all navigation: "Get Started Accessing ACS Data", "View
Popular ACS Tables", "Discover Popular ACS Data Resources". It links out to
`data.census.gov` and to the Bureau's own access points (the API, FTP,
summary files) — so the tutorial does two different things and combines
them:

1. **Scrape this page** for the "Popular ACS Tables" (DP02-DP05) and the
   resource links — legitimate, useful, exactly what a human visiting the
   page would read. `robots.txt` on census.gov only disallows
   `/search-results.html` and a couple of admin paths, so this page is fair
   game.
2. **Pull the actual numbers from the Census Data API** — the Bureau's own
   sanctioned, structured, ToS-compliant way to get real estimates. It's
   what `data.census.gov` calls under the hood, and unlike scraping a
   JavaScript-rendered table, it returns clean JSON that won't break the
   next time the site's front end changes.

## 2. Scrape the popular tables and resource links

Two selectors do all the work, found by inspecting the real class names on
the downloaded page (not guessed): `a.uscb-list__card` for the DP02-DP05
cards, `a.uscb-list-text__item-container` for the resource links.
`fetch_page()` also retries up to 3 times with backoff on network errors —
real connections drop for reasons that have nothing to do with your code.

In [3]:
scraped = scrape_acs_resources.scrape()

def show_table(headers, rows):
    head = "".join(f"<th style='text-align:left;padding:4px 10px;'>{h}</th>" for h in headers)
    body = "".join(
        "<tr>" + "".join(f"<td style='padding:4px 10px;'>{c}</td>" for c in row) + "</tr>"
        for row in rows
    )
    return HTML(f"<table style='border-collapse:collapse;'><thead><tr>{head}</tr></thead><tbody>{body}</tbody></table>")

display(show_table(
    ["Code", "Title", "Description"],
    [(t.code, t.title, t.description) for t in scraped["popular_tables"]],
))
display(show_table(
    ["Resource", "URL"],
    [(l.title, f"<a href='{l.url}' target='_blank'>{l.url}</a>") for l in scraped["resource_links"]],
))

Code,Title,Description
DP02,Selected Social Characteristics,"Disability Status, Educational Attainment, Language Spoken at Home, Veteran Status, and more"
DP03,Selected Economic Characteristics,"Commuting (Journey to Work), Health Insurance Coverage, Income and Earnings, Poverty Status, and more"
DP04,Selected Housing Characteristics,"Computer and Internet Use, House Heating Fuel, Owner/Renter (Tenure), Vehicles Available, and more"
DP05,Demographic and Housing Estimates,"Age and Sex, Group Quarters Population, Hispanic or Latino Origin, Race, and more"


Resource,URL
American Community Survey Data Tables,https://www.census.gov/programs-surveys/acs/data/data-tables.html
American Community Survey Data Tools,https://www.census.gov/programs-surveys/acs/data/data-tools-chart.html
American Community Survey Data via API,https://www.census.gov/programs-surveys/acs/data/data-via-api.html


## 3. Pull real numbers from the Census API

This hits the ACS 5-Year Data Profile endpoint:

```
https://api.census.gov/data/2023/acs/acs5/profile?get=NAME,DP02_0001E&for=state:37&key=...
```

`DP02_0001E` is "Total households" — one variable out of thousands. Browse
the full list for any table code (e.g. `DP03` for economic characteristics)
at `https://api.census.gov/data/2023/acs/acs5/profile/variables.html`.
`STATE_FIPS` in `fetch_acs_data.py` covers North Carolina plus its
neighbors as a regional comparison.

In [4]:
estimates, used_live_api = fetch_acs_data.get_household_estimates()

print("Source:", "live Census API" if used_live_api else "bundled sample data (no CENSUS_API_KEY set, or the call failed)")
display(show_table(
    ["State", fetch_acs_data.VARIABLE_LABEL],
    [(row.name, f"{row.value:,}") for row in estimates],
))

[fetch_acs_data] Live API call failed (Expecting value: line 1 column 1 (char 0)); using bundled sample data.
Source: bundled sample data (no CENSUS_API_KEY set, or the call failed)


State,Total households
North Carolina,"4,147,650"
Georgia,"4,034,282"
Virginia,"3,363,437"
Tennessee,"2,789,587"
South Carolina,"2,117,210"


## 4. Render the HTML report

`generate_report.build_report()` reuses the scrape + fetch calls above,
shapes the data for the Jinja2 template in `templates/report_template.html`,
and writes `index.html`. The chart is hand-built HTML/CSS rather
than a charting library, following a few fixed rules: one hue for the one
series (no legend box needed — the heading already says what's plotted),
thin bars rounded only at the data end, every bar labeled directly at its
tip, and a real second color pass for dark mode rather than an inverted
filter.

In [5]:
generate_report.build_report()
display(IFrame(src="index.html", width="100%", height=650))

[fetch_acs_data] Live API call failed (Expecting value: line 1 column 1 (char 0)); using bundled sample data.
Wrote C:\Users\drdre\Healthy_Vets_Stuff\census_dashboard\output\index.html


## 5. Extend it toward this project's actual focus

This repo's other data (`derived_data/vital_conditions_by_county.csv`, the
`vbh_rscripts/focus_counties.R` work) is North Carolina, county-level. Swap
`for=state:...` for `for=county:*&in=state:37` and one call returns every NC
county — no county FIPS list needed. Try it below (falls back to a friendly
message if the network hiccups, same as the state-level call above).

In [6]:
import requests

try:
    params = {"get": f"NAME,{fetch_acs_data.VARIABLE}", "for": "county:*", "in": "state:37"}
    api_key = __import__("os").environ.get("CENSUS_API_KEY")
    if api_key:
        params["key"] = api_key
    resp = requests.get(fetch_acs_data.BASE_URL, params=params, timeout=15)
    resp.raise_for_status()
    header, *rows = resp.json()
    name_idx, value_idx = header.index("NAME"), header.index(fetch_acs_data.VARIABLE)
    county_rows = sorted(((r[name_idx], int(r[value_idx])) for r in rows), key=lambda r: -r[1])
    print(f"{len(county_rows)} NC counties returned. Top 10 by {fetch_acs_data.VARIABLE_LABEL.lower()}:")
    display(show_table(["County", fetch_acs_data.VARIABLE_LABEL], [(n, f"{v:,}") for n, v in county_rows[:10]]))
except Exception as exc:
    print(f"Live call failed ({exc}). Set CENSUS_API_KEY in .env and re-run this cell — "
          "this is the same request fetch_acs_data.py would make with `for=county:*, in=state:37`.")

Live call failed (Expecting value: line 1 column 1 (char 0)). Set CENSUS_API_KEY in .env and re-run this cell — this is the same request fetch_acs_data.py would make with `for=county:*, in=state:37`.


## 6. Troubleshooting

**`ConnectionResetError [WinError 10054]` or SSL handshake failures.** This
is almost always local network interference — antivirus doing HTTPS
inspection, a corporate proxy, or a VPN — not a problem with the notebook or
with census.gov. Re-run the cell (many of these are intermittent), or
temporarily disable AV/VPN to confirm the cause.

**`Missing Key` in the API response / a JSON decode error.** You don't have
`CENSUS_API_KEY` set (or `.env` isn't being picked up). Confirm `.env`
exists next to this notebook and contains a real key — `fetch_acs_data.py`
calls `load_dotenv()` at import time, which only reads `.env` from the
current working directory.

**The scrape cell returns an empty table.** Census.gov changed its markup.
Re-run the Section 1 cell to see the page's current headings, open devtools,
and update the CSS selectors in `scrape_acs_resources.py` — the normal
maintenance cost of scraping a page you don't control, which is exactly why
Section 3 pushed the actual numbers onto the API instead.

**Wrong kernel / `ModuleNotFoundError`.** Make sure the notebook's kernel
(top-right in VS Code) is set to "Python (census_dashboard venv)", not a
system or different-project Python — that's what Section 0 registered.